# Daily Pipeline on a T4

Profiles one run of the daily forecast: SFNO for three days from a fixed
past date, then the Hot-Dry-Windy Index over the full grid. It answers
how long each stage takes and what share of the run the index takes.
See [docs/daily-risk-map.md](../docs/daily-risk-map.md#pipeline-profile).

Run on Kaggle with **Accelerator: GPU T4 x2** and **Internet: on**. The
notebook uses one GPU and takes about 30 minutes.

The notebook gets the project from GitHub. To use a local copy instead,
upload the repository as a Kaggle dataset named `earth-intelligence`
and add it to the notebook.

Output: `/kaggle/working/daily_pipeline.zip`. Unzip it into
`ml/artifacts/daily_pipeline/`.

In [ ]:
!pip install -q "makani @ git+https://github.com/NVIDIA/makani.git@b38fcb2799d7dbc146fa60459f3f9823394a8bf1" "earth2studio[data,sfno]==0.20.0" python-dotenv ninja 2>&1 | tail -3

In [ ]:
import shutil
import subprocess
from pathlib import Path

REPOSITORY = "https://github.com/khoahonguyen0210/Earth_intelligence.git"
INPUT = Path("/kaggle/input")
PROJECT = Path("/kaggle/working/earth-intelligence")
OUTPUT = Path("/kaggle/working/daily_pipeline")


def uploaded_project():
    for marker in INPUT.rglob("ml/pyproject.toml"):
        return marker.parent.parent
    for archive in INPUT.rglob("earth-intelligence.tar.gz"):
        shutil.unpack_archive(archive, PROJECT)
        return PROJECT
    return None


if not PROJECT.exists():
    source = uploaded_project()
    if source is None:
        subprocess.run(
            ["git", "clone", "--depth", "1", REPOSITORY, str(PROJECT)], check=True
        )
    elif source != PROJECT:
        shutil.copytree(source, PROJECT)

## Download the model once

The first load downloads the SFNO package. Loading it here keeps the
download out of the timed runs.

In [ ]:
!python -c "from earth2studio.models.px import SFNO; SFNO.load_model(SFNO.load_default_package())"

## Runs

All runs use the same cutoff (2025-08-03 10:00 UTC) and ERA5 initial
conditions from ARCO, so they can be compared. Each run loads the model
again. The eager run also saves a `torch.profiler` operator table.

The last two runs show what each memory change does: one keeps only the
latitude-row blocks, and one runs SFNO as released.

In [ ]:
import json

CUTOFF = "2025-08-03T10:00:00+00:00"
RUNS = [
    ("eager", "eager", 64, 32, True),
    ("cuda", "cuda", 64, 32, False),
    ("triton", "triton", 64, 32, False),
    ("torch_compile", "torch_compile", 64, 32, False),
    ("eager_row_blocks_only", "eager", 64, 0, False),
    ("eager_as_released", "eager", 0, 0, False),
]
results = {}
for name, implementation, rows, channels, profile in RUNS:
    command = (
        f"cd {PROJECT} && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True PYTHONPATH=ml/src "
        f"python scripts/map/run_daily_forecast.py --cutoff {CUTOFF} --source arco "
        f"--implementation {implementation} --rows-per-chunk {rows} "
        f"--channels-per-chunk {channels} "
        f"--output {OUTPUT / name}" + (" --profile" if profile else "")
    )
    finished = subprocess.run(command, shell=True, capture_output=True, text=True)
    results[name] = {
        "return_code": finished.returncode,
        "last_lines": (finished.stdout + finished.stderr).splitlines()[-15:],
    }
    print(name, finished.returncode)

OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / "runs.json").write_text(json.dumps(results, indent=2))

In [ ]:
for summary in sorted(OUTPUT.glob("*/summary.json")):
    data = json.loads(summary.read_text())
    print(summary.parent.name, data["total_seconds"], "s", data["peak_memory_gb"], "GB")
    for row in data["stages"]:
        print(f"  {row['stage']:<16} {row['seconds']:>9.3f} s {row['share_of_total']:>8.2%}")

In [ ]:
shutil.make_archive("/kaggle/working/daily_pipeline", "zip", OUTPUT)